# Beat 7: Guided Grad-CAM

**Pergunta:** Dá para ter resolução **e** discriminação?

Este notebook combina a alta resolução do Guided Backpropagation com a discriminação de classe do Grad-CAM através do produto element-wise (`Grad-CAM ⊙ Guided Backprop`).

In [ ]:
import sys
import os
sys.path.append(os.path.abspath('.'))
sys.path.append(os.path.abspath('notebooks'))
import deck
import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from gradcam.models import load_model, preprocess
from gradcam.core import gradcam
from gradcam.saliency import guided_backprop
from gradcam.viz import scale_for_display, upsample, gray_rgb

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = load_model("vgg16", device=device)
model.eval()
print("Ambiente carregado para Beat 7 (Guided Grad-CAM).")

## Calculando o Guided Grad-CAM

Multiplicando o upsampling do Grad-CAM pelo mapa de Guided Backpropagation.

In [ ]:
base_dir = os.path.abspath('.')
if os.path.basename(base_dir) == 'notebooks':
    img_path = "assets/cat_dog.jpg"
else:
    img_path = "notebooks/assets/cat_dog.jpg"
pil_img = Image.open(img_path).convert("RGB")
tensor_img = preprocess(pil_img).to(device)

gcam, class_idx = gradcam(model, tensor_img, class_idx=None, layer=None)
gbp, _ = guided_backprop(model, tensor_img, class_idx=class_idx)

# Upsample Grad-CAM para a resolução 224x224
gcam_up = upsample(gcam, (224, 224))
# Guided Grad-CAM = Grad-CAM (upsampled) * Guided Backprop (magnitude máxima por pixel)
gbp_mag = gbp.abs().amax(dim=0)
guided_gcam = gcam_up * gbp_mag

guided_gcam_scaled = scale_for_display(guided_gcam, high=deck.DISPLAY_HIGH, gamma=deck.DISPLAY_GAMMA)

fig, ax = plt.subplots(figsize=(5, 5), dpi=110)
ax.imshow(gray_rgb(guided_gcam_scaled))
ax.axis("off")
ax.set_title(f"Guided Grad-CAM (Classe ID: {class_idx})")
deck.show(fig)